# 04. Mystery, Espionage & Social Deduction (POV Epistemics, Disguises & Deception)

**Preset:** `InteractiveFictionPreset`

## Why `canonarc` Fits This Domain

Standard RAG and GraphRAG systems query an **omniscient world graph**. In mystery games, TTRPG intrigue, espionage stories, or social deduction benchmarks (*Werewolf / Mafia*), omniscient retrieval ruins the game:
- An unmasked spy's real name leaks into the detective's prompt.
- A villain's secret agenda (`private_state`) or hidden hostility (`stance`) leaks even when they are smiling politely (`facade`).
- A character who was **lied to** (`deceived`) still retrieves the ground-truth secret via vector similarity.

`canonarc` solves all four with **Tier 6 POV Epistemic Projection (`pov_entity_ids`)**:
1. **Disguise & Persona Masking:** `AliasRecord(kind="disguise", known_to=[...], persona_name="...")` automatically masks the entity to `masked:<id>` for anyone not in `known_to`.
2. **Facade vs. Private Stance:** Outward `facade` edges replace hidden `stance` edges for non-self observers, and `private_state` is redacted.
3. **Deceived-POV Vector Isolation:** When an observer has `epistemic="deceived"`, `DefaultEpisodeRanker` embeds their `believed_summary` instead of the ground-truth summary so secret vectors never leak.
4. **Epistemic Updates & Memory Erasure:** Later revelations (`told`) or memory wipes (`forgotten` / `suppressed`) update POV visibility cleanly at the exact step they occur.

In [1]:
from canonarc import (
    AliasRecord,
    EdgeDelta,
    EntityDelta,
    EntityRecord,
    EpisodeRecord,
    InformDelta,
    InteractiveFictionPreset,
    CanonArcEngine,
    StepDelta,
)

engine = CanonArcEngine(":memory:", preset=InteractiveFictionPreset())
print("Mystery / Espionage engine initialized.")

Mystery / Espionage engine initialized.


## Step 1: Seed the Masquerade Cast, Disguise & Facade Edge

- **Lord Vance (`char_vance`)** is infiltrating the banquet disguised as **`"The Velvet Courier"`**. Only Vance and his accomplice **Mira (`char_mira`)** know the disguise (`known_to=["char_vance", "char_mira"]`).
- Vance has a hidden **`private_state`** (`agenda="steal_royal_cipher"`), a hidden hostile **`stance`** toward Detective Clara, and a charming outward **`facade`** edge toward Clara.

In [2]:
engine.register_entity(
    EntityRecord(
        entity_id="char_clara",
        canonical_name="Detective Clara",
        state={"mood": "analytical", "location_id": "loc_ballroom"},
    ),
    aliases=["Clara"],
    step=0,
)
engine.register_entity(
    EntityRecord(
        entity_id="char_vance",
        canonical_name="Lord Vance",
        state={"mood": "poised", "location_id": "loc_ballroom", "disguise": "The Velvet Courier"},
        private_state={"agenda": "steal_royal_cipher", "guilt": "high"},
    ),
    aliases=[
        AliasRecord(
            alias="The Velvet Courier",
            entity_id="char_vance",
            kind="disguise",
            known_to=["char_vance", "char_mira"],
            persona_name="The Velvet Courier",
        )
    ],
    step=0,
)
engine.register_entity(
    EntityRecord(
        entity_id="char_mira",
        canonical_name="Mira",
        state={"mood": "composed", "location_id": "loc_ballroom"},
    ),
    step=0,
)

# Step 5: Vance holds a hostile private stance toward Clara, masked by a warm facade
engine.commit_step(
    step=5,
    delta=StepDelta(
        edges=[
            EdgeDelta(
                source_id="char_vance",
                target_id="char_clara",
                relation_type="stance",
                state={"register": "informal", "tone": "hostile", "trust": -0.9},
            ),
            EdgeDelta(
                source_id="char_vance",
                target_id="char_clara",
                relation_type="facade",
                state={"register": "formal", "tone": "charming", "trust": 0.7},
            ),
        ]
    ),
)
print("Seeded masquerade cast, scoped disguise alias, private_state, and facade edge.")

Seeded masquerade cast, scoped disguise alias, private_state, and facade edge.


## Step 2: Record a Secret Theft & Plant a False Belief (`epistemic="deceived"`)

At **Step 12**, Vance steals the royal cipher from the vault (witnessed by Mira). At **Step 15**, Vance feeds Detective Clara a false cover story:
- `InformDelta(episode_id="ep_step12_cipher_theft", entity_id="char_clara", epistemic="deceived", believed_summary="...")`

In [3]:
engine.commit_step(
    step=12,
    delta=StepDelta(
        episodes=[
            EpisodeRecord(
                episode_id="ep_step12_cipher_theft",
                predicate="theft",
                summary="Lord Vance picked the vault lock and stole the royal cipher while Mira kept watch.",
                participants={"char_vance": "thief", "char_mira": "lookout"},
                witnesses=["char_vance", "char_mira"],
                location_id="loc_vault",
                salience=9.5,
            )
        ]
    ),
)

engine.commit_step(
    step=15,
    delta=StepDelta(
        inform=[
            InformDelta(
                episode_id="ep_step12_cipher_theft",
                entity_id="char_clara",
                epistemic="deceived",
                believed_summary="The royal cipher was accidentally misplaced by the archivist during the banquet.",
            )
        ]
    ),
)
print("Recorded secret theft (Step 12) and planted Clara's false belief (Step 15).")

Recorded secret theft (Step 12) and planted Clara's false belief (Step 15).


## Step 3: Compare 3 Epistemic Lenses at the Exact Same Step (Step 20)

Watch how the **exact same scene** (`"Detective Clara questioned The Velvet Courier about the missing cipher."`) projects completely differently depending on `pov_entity_ids`:
1. **Detective Clara's POV (`pov_entity_ids=["char_clara"]`):**
   - Vance is masked as `The Velvet Courier` (`masked:char_vance`).
   - Vance's `private_state` is redacted.
   - Clara sees Vance's polite `[facade]` edge instead of his hostile `stance`.
   - Clara recalls **only her `believed_summary`** ("misplaced by the archivist")—the real theft never leaks!
2. **Accomplice Mira's POV (`pov_entity_ids=["char_mira"]`):**
   - Mira pierces the disguise (`disguised as: The Velvet Courier (pierced)`).
   - Mira sees the true theft episode.
3. **Omniscient / Game Master POV (`pov_entity_ids=None`):**
   - Sees full ground truth, `private_state`, and both `stance` and `facade` edges.

In [4]:
scene = "Detective Clara questioned The Velvet Courier about the missing cipher."

clara_ctx = engine.retrieve_context(step=20, text=scene, pov_entity_ids=["char_clara"])
mira_ctx = engine.retrieve_context(step=20, text=scene, pov_entity_ids=["char_mira"])
gm_ctx = engine.retrieve_context(step=20, text=scene, pov_entity_ids=None)

print("=== 1. DETECTIVE CLARA'S POV (FOOLED) ===")
print(engine.format_context(clara_ctx))

print("\n=== 2. ACCOMPLICE MIRA'S POV (IN ON THE PLOT) ===")
print(engine.format_context(mira_ctx))

print("\n=== 3. GAME MASTER / OMNISCIENT VIEW ===")
print(engine.format_context(gm_ctx))

# Programmatic anti-metagaming assertions
assert "Lord Vance" not in engine.format_context(clara_ctx)
assert "steal_royal_cipher" not in engine.format_context(clara_ctx)
assert "picked the vault lock" not in engine.format_context(clara_ctx)
assert "misplaced by the archivist" in engine.format_context(clara_ctx)
assert any(e.disguised_as == "The Velvet Courier" for e in mira_ctx.entities)

=== 1. DETECTIVE CLARA'S POV (FOOLED) ===
### Active Scene Narrative Context (Step: 20)

#### Characters Present & State:
- **Detective Clara** (char_clara): mood: analytical
- **The Velvet Courier** (masked:char_vance): mood: neutral, disguised as: The Velvet Courier

#### Episodic Memory (Relevant Past Events):
- [Step 12 | theft @ loc_vault | known to: char_clara, char_mira, char_vance] The royal cipher was accidentally misplaced by the archivist during the banquet.

=== 2. ACCOMPLICE MIRA'S POV (IN ON THE PLOT) ===
### Active Scene Narrative Context (Step: 20)

#### Characters Present & State:
- **Detective Clara** (char_clara): mood: analytical
- **Lord Vance** (char_vance): mood: poised, disguised as: The Velvet Courier (pierced)

#### Interpersonal Stances & Register:
- char_vance → char_clara [facade]: register=formal, tone=charming, trust=0.7

#### Episodic Memory (Relevant Past Events):
- [Step 12 | theft @ loc_vault | known to: char_clara, char_mira, char_vance] Lord Vance p

## Step 4: Unmasking & Truth Reveal (Step 30)

At **Step 30**, Clara catches Vance red-handed: we expand `known_to` on the disguise alias to include `char_clara` and emit `InformDelta(..., epistemic="told")` (or `"direct"`) so Clara's POV now pierces the disguise and sees the true theft.

In [5]:
engine.commit_step(
    step=30,
    delta=StepDelta(
        entities=[
            EntityDelta(
                entity_id="char_vance",
                add_aliases=[
                    AliasRecord(
                        alias="The Velvet Courier",
                        entity_id="char_vance",
                        kind="disguise",
                        known_to=["char_vance", "char_mira", "char_clara"],
                        persona_name="The Velvet Courier",
                    )
                ],
            )
        ],
        inform=[
            InformDelta(
                episode_id="ep_step12_cipher_theft",
                entity_id="char_clara",
                epistemic="told",
            )
        ],
    ),
)

clara_after = engine.retrieve_context(step=35, text=scene, pov_entity_ids=["char_clara"])
print("=== DETECTIVE CLARA'S POV AFTER UNMASKING (STEP 35) ===")
print(engine.format_context(clara_after))

assert "Lord Vance" in engine.format_context(clara_after)
assert "picked the vault lock" in engine.format_context(clara_after)

engine.close()

=== DETECTIVE CLARA'S POV AFTER UNMASKING (STEP 35) ===
### Active Scene Narrative Context (Step: 35)

#### Characters Present & State:
- **Detective Clara** (char_clara): mood: analytical
- **Lord Vance** (char_vance): mood: poised, disguised as: The Velvet Courier (pierced)

#### Interpersonal Stances & Register:
- char_vance → char_clara [facade]: register=formal, tone=charming, trust=0.7

#### Episodic Memory (Relevant Past Events):
- [Step 12 | theft @ loc_vault | known to: char_clara, char_mira, char_vance] Lord Vance picked the vault lock and stole the royal cipher while Mira kept watch.
